In [3]:
# ============================================================
# ✅ BLOCK 1 — INSTALL & IMPORT DEPENDENCIES
# ============================================================

!pip install timm kagglehub -q

import os, glob, shutil
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.nn.functional as F

import timm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device in use:", device)


✅ Device in use: cuda


In [4]:
# ============================================================
# ✅ BLOCK 2 — CONFIGURATION
# ============================================================

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 15
LR_PHASE1 = 1e-4
LR_PHASE2 = 1e-5
MIXUP_ALPHA = 0.2

PROJECT_DIR = "/kaggle/working/SkinDisease_Swin"
os.makedirs(PROJECT_DIR, exist_ok=True)

# ✅ Pretrained Swin Transformer path (your uploaded model)
PRETRAINED_CHECKPOINT = "/kaggle/input/swin-transformerham10000/pytorch/default/1/SwinTransformer_full_model.pth"
print("✅ Using pretrained model from:", PRETRAINED_CHECKPOINT)


✅ Using pretrained model from: /kaggle/input/swin-transformerham10000/pytorch/default/1/SwinTransformer_full_model.pth


In [5]:
# ============================================================
# ✅ BLOCK 3 — LOAD ISIC 2019 DATASET (FOLDER-BASED)
# ============================================================

import os
import pandas as pd
import kagglehub

print("Loading ISIC 2019 dataset...")
ISIC_PATH = kagglehub.dataset_download("salviohexia/isic-2019-skin-lesion-images-for-classification")
print("✅ Dataset located at:", ISIC_PATH)

# Paths
metadata_path = os.path.join(ISIC_PATH, "ISIC_2019_Training_GroundTruth.csv")
metadata = pd.read_csv(metadata_path)

# Create single label column (convert one-hot encoding)
label_cols = [c for c in metadata.columns if c != "image"]
metadata["label"] = metadata[label_cols].idxmax(axis=1)

# Remove unwanted classes (SCC or UNK if needed)
exclude_classes = ["UNK", "SCC"]
metadata = metadata[~metadata["label"].isin(exclude_classes)]

print("✅ Classes after filtering:\n", metadata["label"].value_counts())


Loading ISIC 2019 dataset...
✅ Dataset located at: /kaggle/input/isic-2019-skin-lesion-images-for-classification
✅ Classes after filtering:
 label
NV      12875
MEL      4522
BCC      3323
BKL      2624
AK        867
VASC      253
DF        239
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(metadata, test_size=0.2, random_state=42, stratify=metadata["label"])
class_names = sorted(metadata["label"].unique())
num_classes = len(class_names)

print(f"✅ Total Classes: {num_classes}")
print(f"✅ Class Names: {class_names}")


✅ Total Classes: 7
✅ Class Names: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']


In [7]:
# ============================================================
# ✅ BLOCK 5 — DEFINE CUSTOM DATASET (CLASS FOLDER AWARE)
# ============================================================

from torch.utils.data import Dataset
from PIL import Image

class ISICDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.label_to_idx = {label: i for i, label in enumerate(sorted(df["label"].unique()))}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['image']
        label_name = row['label']

        # ✅ Try to find the image inside the correct folder (e.g., /AK/, /BCC/, /MEL/, etc.)
        img_path = os.path.join(self.root_dir, label_name, f"{image_id}.jpg")

        # Double-check file existence
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        image = Image.open(img_path).convert("RGB")
        label = self.label_to_idx[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label


In [8]:
# ============================================================
# ✅ BLOCK 6 — TRANSFORMS & DATALOADERS
# ============================================================

from torchvision import transforms as T
from torch.utils.data import DataLoader

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# ✅ Point directly to main dataset root (no /images)
image_dir = ISIC_PATH

train_dataset = ISICDataset(train_df, image_dir, transform=train_transform)
val_dataset = ISICDataset(val_df, image_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("✅ Dataloaders ready!")
print(f"✅ Total training samples: {len(train_dataset)}")
print(f"✅ Total validation samples: {len(val_dataset)}")


✅ Dataloaders ready!
✅ Total training samples: 19762
✅ Total validation samples: 4941


In [9]:
# ============================================================
# ✅ BLOCK 7 — LOAD PRETRAINED SWIN TRANSFORMER (FULL MODEL)
# ============================================================

from timm.models.swin_transformer import SwinTransformer
import torch.serialization

# Allow SwinTransformer class for safe unpickling
torch.serialization.add_safe_globals([SwinTransformer])

# Load the full model directly (since it was saved with torch.save(model))
checkpoint = torch.load(
    "/kaggle/input/swin-transformerham10000/pytorch/default/1/SwinTransformer_full_model.pth",
    map_location=device,
    weights_only=False
)

# If the checkpoint is already a model instance
if isinstance(checkpoint, nn.Module):
    model = checkpoint
    print("✅ Loaded full Swin Transformer model directly from checkpoint.")
else:
    # If it’s a dict, extract state_dict
    model = timm.create_model("swin_base_patch4_window7_224", pretrained=False, num_classes=num_classes)
    model.load_state_dict(checkpoint, strict=False)
    print("✅ Loaded weights from checkpoint dictionary.")

model = model.to(device)

# Replace classification head for ISIC 2019 (8 classes)
in_features = model.head.in_features
model.head = nn.Linear(in_features, num_classes)
model.head.to(device)

print("✅ Swin Transformer ready for ISIC 2019 fine-tuning.")


✅ Loaded full Swin Transformer model directly from checkpoint.
✅ Swin Transformer ready for ISIC 2019 fine-tuning.


In [10]:
# ============================================================
# ✅ BLOCK 8 (MODIFIED) — PHASE 1 TRAINING (FROZEN BACKBONE, NO AMP)
# ============================================================

# Freeze all layers except classification head
for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

# Ensure model is on device and parameters are float32
model = model.to(device).float()

# Loss and optimizer (train only head params)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.head.parameters(), lr=LR_PHASE1)

# Training & validation loops (NO AMP)
def train_one_epoch_fp32(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc="Training", leave=False):
        imgs = imgs.to(device).float()   # ensure float32
        labels = labels.to(device)
        optimizer.zero_grad()

        outputs = model(imgs)             # forward (float32)
        # If model returns spatial map (B, C, H, W), reduce to (B, C)
        if outputs.ndim > 2:
            outputs = outputs.mean(dim=(2,3))
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate_fp32(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Validation", leave=False):
            imgs = imgs.to(device).float()
            labels = labels.to(device)
            outputs = model(imgs)
            if outputs.ndim > 2:
                outputs = outputs.mean(dim=(2,3))
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


# Train Phase 1 (no AMP)
best_acc = 0.0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(EPOCHS_PHASE1):
    print(f"\nEpoch [{epoch+1}/{EPOCHS_PHASE1}] — Phase 1")
    train_loss, train_acc = train_one_epoch_fp32(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate_fp32(model, val_loader, criterion)

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accs.append(train_acc); val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(PROJECT_DIR, "phase1_best.pth"))
        print("✅ Saved new best model for Phase 1!")

print("✅ Phase 1 finished. Best val acc: {:.4f}".format(best_acc))



Epoch [1/10] — Phase 1


Train Loss: 1.6104, Train Acc: 0.4580
Val Loss:   1.4297, Val Acc:   0.5351
✅ Saved new best model for Phase 1!

Epoch [2/10] — Phase 1


Train Loss: 1.3582, Train Acc: 0.5462
Val Loss:   1.3075, Val Acc:   0.5588
✅ Saved new best model for Phase 1!

Epoch [3/10] — Phase 1


Train Loss: 1.2826, Train Acc: 0.5644
Val Loss:   1.2496, Val Acc:   0.5732
✅ Saved new best model for Phase 1!

Epoch [4/10] — Phase 1


Train Loss: 1.2385, Train Acc: 0.5749
Val Loss:   1.2160, Val Acc:   0.5823
✅ Saved new best model for Phase 1!

Epoch [5/10] — Phase 1


...

Train Loss: 1.1925, Train Acc: 0.5875
Val Loss:   1.1721, Val Acc:   0.5956
✅ Saved new best model for Phase 1!

Epoch [7/10] — Phase 1


Train Loss: 1.1730, Train Acc: 0.5944
Val Loss:   1.1575, Val Acc:   0.5991
✅ Saved new best model for Phase 1!

Epoch [8/10] — Phase 1


Train Loss: 1.1584, Train Acc: 0.5973
Val Loss:   1.1475, Val Acc:   0.6041
✅ Saved new best model for Phase 1!

Epoch [9/10] — Phase 1


Train Loss: 1.1541, Train Acc: 0.6015
Val Loss:   1.1399, Val Acc:   0.6066
✅ Saved new best model for Phase 1!

Epoch [10/10] — Phase 1


Train Loss: 1.1451, Train Acc: 0.6009
Val Loss:   1.1323, Val Acc:   0.6126
✅ Saved new best model for Phase 1!
✅ Phase 1 finished. Best val acc: 0.6126


In [11]:
# ============================================================
# ✅ BLOCK 9 — PHASE 2 TRAINING (UNFREEZE BACKBONE)
# ============================================================

# Unfreeze the entire model for fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Load best weights from Phase 1
phase1_path = os.path.join(PROJECT_DIR, "phase1_best.pth")
if os.path.exists(phase1_path):
    model.load_state_dict(torch.load(phase1_path, map_location=device))
    print("✅ Loaded best Phase 1 model for fine-tuning.")

# Define new optimizer (lower LR)
optimizer = optim.Adam(model.parameters(), lr=LR_PHASE2)
criterion = nn.CrossEntropyLoss()

# Train & validate using same functions (float32 safe)
best_acc_phase2 = 0.0

for epoch in range(EPOCHS_PHASE2):
    print(f"\nEpoch [{epoch+1}/{EPOCHS_PHASE2}] — Phase 2 (Fine-tuning)")
    train_loss, train_acc = train_one_epoch_fp32(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate_fp32(model, val_loader, criterion)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")

    if val_acc > best_acc_phase2:
        best_acc_phase2 = val_acc
        torch.save(model.state_dict(), os.path.join(PROJECT_DIR, "swin_isic_best.pth"))
        print("✅ Saved new best fine-tuned model (Phase 2)!")

print("🎯 Phase 2 complete. Best validation accuracy: {:.4f}".format(best_acc_phase2))


✅ Loaded best Phase 1 model for fine-tuning.

Epoch [1/15] — Phase 2 (Fine-tuning)


Train Loss: 0.7804, Train Acc: 0.7268
Val Loss:   0.6164, Val Acc:   0.7857
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [2/15] — Phase 2 (Fine-tuning)


Train Loss: 0.5460, Train Acc: 0.8078
Val Loss:   0.5107, Val Acc:   0.8221
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [3/15] — Phase 2 (Fine-tuning)


Train Loss: 0.4454, Train Acc: 0.8417
Val Loss:   0.4633, Val Acc:   0.8377
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [4/15] — Phase 2 (Fine-tuning)


Train Loss: 0.3622, Train Acc: 0.8694
Val Loss:   0.4592, Val Acc:   0.8454
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [5/15] — Phase 2 (Fine-tuning)


...

Train Loss: 0.1031, Train Acc: 0.9645
Val Loss:   0.4635, Val Acc:   0.8786
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [12/15] — Phase 2 (Fine-tuning)


Train Loss: 0.0871, Train Acc: 0.9703
Val Loss:   0.4411, Val Acc:   0.8838
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [13/15] — Phase 2 (Fine-tuning)


Train Loss: 0.0774, Train Acc: 0.9733
Val Loss:   0.4615, Val Acc:   0.8867
✅ Saved new best fine-tuned model (Phase 2)!

Epoch [14/15] — Phase 2 (Fine-tuning)


Train Loss: 0.0635, Train Acc: 0.9785
Val Loss:   0.5301, Val Acc:   0.8790

Epoch [15/15] — Phase 2 (Fine-tuning)


Train Loss: 0.0599, Train Acc: 0.9804
Val Loss:   0.4760, Val Acc:   0.8861
🎯 Phase 2 complete. Best validation accuracy: 0.8867


In [12]:
import torch
import os

# Folder to save models
SAVE_DIR = PROJECT_DIR  # you can change this

# 1️⃣ Save only the weights (state_dict) — optional, you already did this
weights_path = os.path.join(SAVE_DIR, "swin_isic_phase2_weights.pth")
torch.save(model.state_dict(), weights_path)
print(f"✅ Model weights saved at: {weights_path}")

# 2️⃣ Save the entire model (architecture + weights)
full_model_path = os.path.join(SAVE_DIR, "swin_isic_phase2_full_model.pth")
torch.save(model, full_model_path)
print(f"✅ Full model saved at: {full_model_path}")


✅ Model weights saved at: /kaggle/working/SkinDisease_Swin/swin_isic_phase2_weights.pth
✅ Full model saved at: /kaggle/working/SkinDisease_Swin/swin_isic_phase2_full_model.pth


In [13]:
import torch
import os

# ✅ Save models in Kaggle output directory
SAVE_DIR = "/kaggle/working/"

# 1️⃣ Save only the weights (state_dict)
weights_path = os.path.join(SAVE_DIR, "swin_isic_phase2_weights.pth")
torch.save(model.state_dict(), weights_path)
print(f"✅ Model weights saved at: {weights_path}")

# 2️⃣ Save the entire model (architecture + weights)
full_model_path = os.path.join(SAVE_DIR, "swin_isic_phase2_full_model.pth")
torch.save(model, full_model_path)
print(f"✅ Full model saved at: {full_model_path}")


✅ Model weights saved at: /kaggle/working/swin_isic_phase2_weights.pth
✅ Full model saved at: /kaggle/working/swin_isic_phase2_full_model.pth
